# Stage 2 Analysis

Stage 2 models weekly interruption risk after the learner has already passed the initial Stage 1 observation window. Each example uses the previous `SEQUENCE_LENGTH` weeks to predict whether the next labeled week is a dropout point.

Unlike Stage 1, this stage keeps the temporal structure of learner behavior. The model can therefore learn patterns such as sustained engagement, declining activity, irregular usage, or seasonal pauses instead of relying only on static early aggregates.


## Setup


In [1]:
import os
from pathlib import Path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd

import torch

PROJECT_ROOT = Path.cwd()

from src.preprocess import build_stage2_dataset
from src import model_utils as lstm
from src import sequence_utils as seq

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
SEQUENCE_LENGTH = 12
BATCH_SIZE = 64
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


## Build Dataset

The Stage 2 dataset is built with `build_stage2_dataset`. It creates one row per learner-week, with behavioral features such as activity, sessions, transactions, correctness, topic breadth, and calendar indicators.

The label `is_dropout_point` marks weeks that precede an observed interruption. It is a local sequence label, not a global statement that the learner permanently left the platform.


In [2]:
stage2 = build_stage2_dataset(DATA_DIR)
df_full = stage2.df_full
feature_cols = stage2.feature_cols
users_clean = stage2.users
user_features = stage2.user_features

print(f"timeline rows: {len(df_full):,}")
print(f"users: {df_full['user_id'].nunique():,}")
print(f"features: {len(feature_cols)}")
print(f"labeled rows: {df_full['is_dropout_point'].notna().sum():,}")
print(feature_cols)

df_full.head()


timeline rows: 408,377
users: 22,470
features: 17
labeled rows: 342,844
['n_events', 'n_active_days', 'mean_hour', 'n_click_events', 'n_view_events', 'n_sessions', 'n_topics_event', 'n_transactions', 'correct_rate', 'partial_rate', 'mean_evaluation_score', 'avg_response_time', 'n_documents', 'n_topics_transaction', 'activity_score', 'year', 'day']


,user_id,relative_week,n_events,n_active_days,mean_hour,n_click_events,n_view_events,n_sessions,n_topics_event,n_transactions,...,avg_response_time,n_documents,n_topics_transaction,activity_score,week_start,year,day,is_active,is_dropout_point,is_summer
0,387604,0,2.0,2.0,9.0,0.0,2.0,0.0,0.0,2.0,...,0.0,1.0,0.0,6.0,2021-05-22 05:12:11.249,2021,142,1,0.0,False
1,387604,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-05-29 05:12:11.249,2021,149,0,0.0,False
2,387604,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-06-05 05:12:11.249,2021,156,0,0.0,False
3,387604,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-06-12 05:12:11.249,2021,163,0,0.0,False
4,387604,4,7.0,1.0,8.0,4.0,3.0,1.0,1.0,0.0,...,0.0,0.0,0.0,14.0,2021-06-19 05:12:11.249,2021,170,1,0.0,False


## Split And Scale

The split is performed by user, not by row. This avoids leakage where weeks from the same learner appear in both training and testing. Feature scaling is fit on the training users only, then applied to validation and test users.

This design makes the evaluation closer to the real task: predicting future behavior for learners not seen during training.


In [ ]:
train_df, val_df, test_df, scaler = seq.split_and_scale_by_users(
    df_full,
    feature_cols,
    test_size=0.15,
    val_size=0.20,
    random_state=RANDOM_STATE,
)
print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)


train: (275745, 23)
val:   (70920, 23)
test:  (61712, 23)


## Sequence Datasets

Weekly rows are converted into fixed-length sequences. For each labeled week, the input is the previous `SEQUENCE_LENGTH` weeks and the target is the dropout-point label of the following week.

This representation lets the LSTM use the order of behavioral changes, which is precisely the information lost in a static aggregate table.


In [ ]:
loaders, _ = lstm.make_sequence_loaders(
    train_df,
    val_df,
    test_df,
    feature_cols,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
)
train_loader, val_loader, test_loader = loaders
print("sequence batches:", len(train_loader), len(val_loader), len(test_loader))


sequence batches: 2250 591 506


## Experiment Runner

The experiment runner keeps the repeated training procedure consistent across settings: user split, scaling, sequence construction, model initialization, early stopping, validation-threshold tuning, and test evaluation.

Keeping this orchestration in one place makes the comparison between experiments easier to interpret.


In [ ]:
def run_experiment(name, cols, demographics=False, drop_summer_test=False):
    local_train_df, local_val_df, local_test_df, _ = seq.split_and_scale_by_users(
        df_full,
        cols,
        test_size=0.15,
        val_size=0.20,
        random_state=RANDOM_STATE,
    )
    loaders, demo_sizes = lstm.make_sequence_loaders(
        local_train_df,
        local_val_df,
        local_test_df,
        cols,
        sequence_length=SEQUENCE_LENGTH,
        batch_size=BATCH_SIZE,
        users=users_clean,
        demographics=demographics,
        drop_summer_test=drop_summer_test,
    )
    train_loader, val_loader, test_loader = loaders

    if demographics:
        model = lstm.LSTMWithDemographics(
            input_size=len(cols),
            n_gender=demo_sizes["n_gender"],
            n_canton=demo_sizes["n_canton"],
            n_class=demo_sizes["n_class"],
        ).to(device)
    else:
        model = lstm.LSTMModel(input_size=len(cols)).to(device)

    model, history = lstm.train_lstm_model(
        model,
        train_loader,
        val_loader,
        device,
        epochs=EPOCHS,
        patience=PATIENCE,
        learning_rate=LEARNING_RATE,
        demographics=demographics,
    )
    metrics, test_prob, test_y = lstm.evaluate_lstm_model(
        model,
        val_loader,
        test_loader,
        device,
        demographics=demographics,
    )

    result = {
        "experiment": name,
        "n_features": len(cols),
        "demographics": demographics,
        "drop_summer_test": drop_summer_test,
        **metrics,
    }
    return result, history, model, test_prob, test_y


## Experiments

The experiments reproduce the main questions from the older Stage 2 notebook.

- The full weekly-feature model tests the predictive value of behavioral trajectories.
- The no-summer-test variant checks whether performance is inflated by easy seasonal patterns.
- The time-only model estimates how much can be predicted from calendar information alone.
- The demographics variant tests whether profile information adds signal beyond weekly behavior.

Together these settings separate behavioral prediction from seasonal and demographic shortcuts.


In [8]:
experiments = [
    {
        "name": "all_weekly_features",
        "cols": feature_cols,
        "demographics": False,
        "drop_summer_test": False,
    },
    {
        "name": "all_weekly_features_no_summer_test",
        "cols": feature_cols,
        "demographics": False,
        "drop_summer_test": True,
    },
    {
        "name": "time_only",
        "cols": ["year", "day"],
        "demographics": False,
        "drop_summer_test": False,
    },
    {
        "name": "weekly_features_with_demographics",
        "cols": feature_cols,
        "demographics": True,
        "drop_summer_test": False,
    },
]

results = []
histories = {}
models = {}

for spec in experiments:
    print()
    print("=" * 80)
    print(spec["name"])
    result, history, model, test_prob, test_y = run_experiment(**spec)
    results.append(result)
    histories[spec["name"]] = history
    models[spec["name"]] = model
    print(result)

results_df = pd.DataFrame(results)
results_df



all_weekly_features
epoch 01 | train_loss=0.4513 | val_f1=0.7973 | threshold=0.133
epoch 02 | train_loss=0.4252 | val_f1=0.8063 | threshold=0.410
epoch 03 | train_loss=0.4159 | val_f1=0.8116 | threshold=0.345
epoch 04 | train_loss=0.4082 | val_f1=0.8128 | threshold=0.378
epoch 05 | train_loss=0.4012 | val_f1=0.8150 | threshold=0.361
epoch 06 | train_loss=0.3946 | val_f1=0.8189 | threshold=0.427
epoch 07 | train_loss=0.3890 | val_f1=0.8275 | threshold=0.427
epoch 08 | train_loss=0.3845 | val_f1=0.8251 | threshold=0.459
epoch 09 | train_loss=0.3794 | val_f1=0.8211 | threshold=0.394
epoch 10 | train_loss=0.3743 | val_f1=0.8219 | threshold=0.394
epoch 11 | train_loss=0.3706 | val_f1=0.8247 | threshold=0.361
epoch 12 | train_loss=0.3652 | val_f1=0.8244 | threshold=0.394
early stopping
{'experiment': 'all_weekly_features', 'n_features': 17, 'demographics': False, 'drop_summer_test': False, 'roc_auc': 0.8848601387472863, 'f1': 0.8224609155325611, 'precision': 0.7335975120385233, 'recall': 0.

,experiment,n_features,demographics,drop_summer_test,roc_auc,f1,precision,recall,threshold,val_f1
0,all_weekly_features,17,False,False,0.884860,0.822461,0.733598,0.935820,0.426531,0.827536
1,all_weekly_features_no_summer_test,17,False,True,0.833331,0.826026,0.746677,0.924245,0.426531,0.829372
2,time_only,2,False,False,0.818266,0.795895,0.662052,0.997568,0.263265,0.800536
3,weekly_features_with_demographics,17,True,False,0.912045,0.841113,0.763176,0.936780,0.328571,0.844108


## Save Results

The result table stores the final test metrics for each experimental setting, while the history files store the training curves. These files are useful for reporting and for checking whether a result came from stable training or from a noisy run.


In [9]:
results_df.to_csv(OUTPUT_DIR / "stage2_lstm_results.csv", index=False)
for name, history in histories.items():
    history.to_csv(OUTPUT_DIR / f"stage2_history_{name}.csv", index=False)

results_df.sort_values("f1", ascending=False)


,experiment,n_features,demographics,drop_summer_test,roc_auc,f1,precision,recall,threshold,val_f1
3,weekly_features_with_demographics,17,True,False,0.912045,0.841113,0.763176,0.936780,0.328571,0.844108
1,all_weekly_features_no_summer_test,17,False,True,0.833331,0.826026,0.746677,0.924245,0.426531,0.829372
0,all_weekly_features,17,False,False,0.884860,0.822461,0.733598,0.935820,0.426531,0.827536
2,time_only,2,False,False,0.818266,0.795895,0.662052,0.997568,0.263265,0.800536
